# 04 · Pilot: temperature sweep with two new generator families (design doc §8.8.9)

**How to run**: pick a Colab server with a GPU (a T4 is enough) and click "Run All"; about 3.5 hours. If the connection drops, click "Run All" again; finished parts are skipped. Results go to `DATA_DIR/04_pilot_new_generators/`.

**What it computes**: the design of notebook 03 with two base models from families other than Qwen: phi-2 (Microsoft, 2.7B) and SmolLM2-1.7B (Hugging Face). The same 100 human texts, first 30 tokens and six temperatures (0.6 to 1.2, top-p 1.0); GPT-2 XL and each model itself score the continuations. The goal is to check that "the margin predicts the failure temperature" holds beyond Qwen.

In [ ]:
# 1. Check for a GPU.
import torch
assert torch.cuda.is_available(), "No GPU: switch the Colab server to a GPU"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GB")

In [ ]:
# 2. Mount Google Drive and switch to the data folder.
import os
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/curvature-margin/data"  # folder with the layout in data/README.md
%cd $DATA_DIR
os.makedirs("04_pilot_new_generators", exist_ok=True)
os.environ["HF_HOME"] = "/content/hf"

In [ ]:
# 3. Install packages and load the human texts of notebook 01.
%pip install -q -U transformers huggingface-hub accelerate
import json, gc, glob, shutil, traceback, time
import numpy as np, pandas as pd
humans = json.load(open("01_pilot_decoding/humans.json"))
print(len(humans), "human texts")

In [ ]:
# 4. Continuations: two generators x six temperatures, top-p fixed at 1.0. Each finished condition is saved and skipped on a rerun.
import token_info
GENS = ["microsoft/phi-2", "HuggingFaceTB/SmolLM2-1.7B"]
TEMPS = [0.6, 0.8, 0.9, 1.0, 1.1, 1.2]
PREFIX = 30

for gen in GENS:
    tag = gen.split("/")[1]
    for temp in TEMPS:
        path = f"04_pilot_new_generators/gen_{tag}_T{temp}.json"
        if os.path.exists(path):
            print("Already done, skipping:", path); continue
        engine = None
        try:
            token_info.SETTINGS.update(scoring_model=gen, sampling_model=gen, temperature=temp, top_p=1.0, do_sample=True,
                                       prefix_tokens=PREFIX, paraphrase_enabled=False, save_token_features=False,
                                       cache_dir="/content/hf", seed=42)
            engine = token_info.TokenInfo()
            rows, start = [], time.time()
            for h in humans:
                try:
                    r = engine.continue_text(h["text"])
                except (ValueError, RuntimeError) as e:
                    print("Skipping text", h["id"], ":", e); continue
                ids = engine.encode(h["text"])
                rows.append(dict(id=h["id"], domain=h["domain"], model=gen, decoding=f"T{temp}", prefix=r["prefix"],
                                 ai=r["continuation"],
                                 human=engine.scoring_tokenizer.decode(ids[PREFIX:], clean_up_tokenization_spaces=False)))
                if len(rows) % 25 == 0:
                    print(f"  {tag} T={temp}: {len(rows)}/{len(humans)} texts, {(time.time() - start) / 60:.0f} min so far", flush=True)
            json.dump(rows, open(path, "w"), ensure_ascii=False)
            print(tag, "T =", temp, "done:", len(rows), "texts")
        except Exception:
            traceback.print_exc(); print("This condition failed, skipping:", gen, temp)
        finally:
            engine = None; gc.collect(); torch.cuda.empty_cache()
    shutil.rmtree(f"/content/hf/models--{gen.replace('/', '--')}", ignore_errors=True)

In [ ]:
# 5. Scoring: GPT-2 XL scores all texts of both models; each generator also scores its own texts.
from transformers import AutoTokenizer, AutoModelForCausalLM

def score_texts(scorer_name, records, out_path):
    if os.path.exists(out_path):
        print("Already done, skipping:", out_path); return
    tok = AutoTokenizer.from_pretrained(scorer_name)
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16
    model = AutoModelForCausalLM.from_pretrained(scorer_name, dtype=dtype).cuda().eval()
    out = []
    with torch.no_grad():
        for i, r in enumerate(records):
            k = len(tok(r["prefix"])["input_ids"])
            for who in ("human", "ai"):
                ids = tok(r["prefix"] + r[who], truncation=True, max_length=512)["input_ids"]
                if len(ids) - k < 10:
                    continue
                x = torch.tensor([ids]).cuda()
                lp = torch.log_softmax(model(x).logits[0, :-1].float(), -1)
                p = lp.exp(); e1 = (p * lp).sum(-1)
                rows = [-lp.gather(1, x[0, 1:, None])[:, 0], -e1, (p * lp * lp).sum(-1) - e1 ** 2,
                        p.topk(10, -1).values.sum(-1)]
                out.append(dict(id=r["id"], domain=r["domain"], model=r["model"], decoding=r["decoding"],
                                scorer=scorer_name, who=who,
                                arr=torch.stack(rows).cpu().numpy().astype(np.float32)[:, k - 1:]))
            if i % 100 == 0:
                print(" ", scorer_name, i, "/", len(records), flush=True)
    pd.DataFrame(out).to_pickle(out_path)
    del model; gc.collect(); torch.cuda.empty_cache()
    shutil.rmtree(f"/content/hf/models--{scorer_name.replace('/', '--')}", ignore_errors=True)

recs = {gen: sum((json.load(open(f)) for f in sorted(glob.glob(f"04_pilot_new_generators/gen_{gen.split('/')[1]}_T*.json"))), []) for gen in GENS}
score_texts("gpt2-xl", recs[GENS[0]] + recs[GENS[1]], "04_pilot_new_generators/scores_gpt2-xl.pkl")
for gen in GENS:
    score_texts(gen, recs[gen], f"04_pilot_new_generators/scores_{gen.split('/')[1]}.pkl")

In [ ]:
# 6. Quick summary: margin and AUROC per (generator, scorer, temperature). The full analysis is scripts/a11_new_generators.py.
from sklearn.metrics import roc_auc_score
s = pd.concat([pd.read_pickle(f) for f in sorted(glob.glob("04_pilot_new_generators/scores_*.pkl"))], ignore_index=True)
s["surprisal"] = [a[0].mean() for a in s.arr]
s["fdg"] = [(a[1].sum() - a[0].sum()) / np.sqrt(a[2].sum()) for a in s.arr]
s["y"] = (s.who == "ai").astype(int)
s["T"] = s.decoding.str[1:].astype(float)
def per(g):
    h, ai = g[g.y == 0], g[g.y == 1]
    return pd.Series({"surprisal_margin": h.surprisal.mean() - ai.surprisal.mean(), "AUROC": roc_auc_score(g.y, g.fdg)})
table = s.groupby(["model", "scorer", "T"]).apply(per)
table.to_csv("04_pilot_new_generators/summary.csv")
print(table.round(3))

In [ ]:
# 7. Flush unsynced files back to Google Drive.
%cd /content
drive.flush_and_unmount()
print("Done: results are in DATA_DIR/04_pilot_new_generators/")